# Anchor Refinery
Created by Quillan Shimp with the use of Claude. Images from Legacy Survey.  

### How to Use
Ensure cutout_vetter.py and vetter_data.py are in this directory.  
Select SGAML (or NERSC Python) kernel.  
Start the vetting program by running all cells.  
Input your username and select galaxy size bin.  
Choose to inject new galaxies as anchors or to vet our working anchor sample.  
Use the morphology dropdown to change galaxy type, use the style buttons to change image type.  
Select any number of galaxies. Use buttons to assign an alternate morphology or bad anchor attribute.  
Click the galaxy again to deselect.  
Click affirm all to assign galaxies on page as the given type.  
You may change the number of galaxies per page and number of columns in the Settings cell.  
You may click save to save your progress and exit to end your session.  
### Advice
Click save intermittently while vetting (I do so every 250 galaxies).  
If you have notes, write before assigning morphology.  
If memory overloads during a session and the page refreshes, don't panic. The tool will reload where you left off and you can save from there.  
If you think of any improvement, let me know!  
### The Sample
Vetting mode draws from a master file of vetter classifications.  
Selection mode draws from SGA-2025 v1.0  

In [1]:
# for debugging cutout_vetter
%load_ext autoreload
%autoreload 2

In [2]:
import os
import glob
import time
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import ipywidgets as widgets

from IPython.display import display, clear_output

%matplotlib widget

from vetlib.vetter_data import (
    VettingStore,
    SizeBinSource,
    MODE_UNVETTED,
    MODE_AGREED,
    confusion_matrix_report,
    anchor_progress,
    vetting_depth
)

from vetlib.cutout_vetter import ReviewerLogin
from vetlib.jpg_service import JpgService

In [3]:
# Legacy Survey Viewer image products
JPEG_ROOT = Path("/global/cfs/cdirs/desicollab/users/qshimp/cutouts/SGA-2025")
IMAGE_DIR = JPEG_ROOT / "Image"
MODEL_DIR = JPEG_ROOT / "Model"
RESIDUAL_DIR = JPEG_ROOT / "Residual"

# Shared output directory for multi-user functionality.
MASTER_DIR = Path("/global/cfs/cdirs/desicollab/users/qshimp/anchors")

# SSL HDF5 cutouts
SSL_DIR = Path("/global/cfs/cdirs/desicollab/users/ioannis/SGA/2025/ssl")

# ML classifications by size bin. Work in progress
ML_CATALOGS = {
    "0-39arcsec": "/global/cfs/cdirs/desicollab/users/qshimp/morph_catalogs/bin0/SGA2025beta_v1.0.csv",
    "40-79arcsec": "/global/cfs/cdirs/desicollab/users/qshimp/morph_catalogs/bin1/SGA2025beta_v1.0.csv",
    #"80-119arcsec": "/path/to/ml_catalog_large.csv",
    #"120-159arcesc": "/path/to/ml_catalog_extralarge.csv",
    #"160+arcsec": "/path/to/ml_catalog_extraextralarge.csv"
}

# Full SGA Catalog (For SGAID, RA, DEC, and D26).
SGA_CATALOG_PATH = Path("/global/cfs/cdirs/desicollab/users/qshimp/catalogs/SGA2025_v1.0.csv")

# Display settings
N_COLS = 10
N_PER_PAGE = 50
N_WORKERS = 8

In [4]:
H5_ROOT = "/global/cfs/cdirs/desicollab/users/qshimp/h5py_files"

size_bins = [
    SizeBinSource(label=label, ml_catalog=path, jpg_pix=152, coadd_dir=f"{H5_ROOT}/bin{i}") # Change when adding larger gals
    for i, (label, path) in enumerate(ML_CATALOGS.items())
]

for source in size_bins:
    t = time.time()
    ids = pd.read_csv(source.ml_catalog, usecols=["SGAID"], low_memory=False)["SGAID"]
    print(f"{source.label}: {len(ids):,} rows; {ids.nunique():,} unique SGAIDs  ({time.time()-t:.0f}s)", flush=True)

t = time.time()
store = VettingStore(master_dir=MASTER_DIR, catalog=SGA_CATALOG_PATH,
                     ml_id_col="SGAID", ml_morph_col="Galaxy_Type")
print(f"Catalog loaded ({time.time()-t:.0f}s).")
master = store.rebuild_master()

0-39arcsec: 212,331 rows; 212,331 unique SGAIDs  (2s)
40-79arcsec: 115,308 rows; 115,308 unique SGAIDs  (0s)
Catalog loaded (22s).


In [5]:
print(f"Vetting directory: {store.master_dir}")
print(f"Vetter files: {store.list_vetters()}")
print(f"Master table: {store.master_path}")
display(master.head())

Vetting directory: /global/cfs/cdirs/desicollab/users/qshimp/anchors
Vetter files: ['nravi', 'qshimp', 's']
Master table: /global/cfs/cdirs/desicollab/users/qshimp/anchors/vetting_master.csv


,SGAID,RA,DEC,D26,nravi,qshimp,s
0,570,43.685625,-11.861502,0.645587,NaN,Irregular,Irregular
1,663,52.790040,2.160897,0.908740,NaN,Irregular,Irregular
2,902,87.898192,-60.961305,0.528081,NaN,Irregular,Bad anchor
3,934,94.202777,-14.868604,0.488316,NaN,Irregular,Irregular
4,1462,154.459240,-33.919423,0.864963,NaN,Irregular,Irregular


In [6]:
# Taken from /global/homes/q/qshimp/SGA/doc/tutorials/SGA-ssl.ipynb
def build_cutout_index(ssl_dir):
    """
    Return {(region, sgaid): (hdf5_path, row_index)}
    for fast image retrieval.
    """
    files = sorted(glob.glob(os.path.join(ssl_dir, "ssl-cutouts-dr11-*.hdf5")))
    if not files:
        raise FileNotFoundError(f"No cutout files found in {ssl_dir}")

    index = {}
    for f in files:
        filename = os.path.basename(f)
        
        if "dr11-south" in filename:
            region = "dr11-south"
        elif "dr11-north" in filename:
            region = "dr11-north"
        else:
            raise ValueError(f"Could not determine region from filename: {filename}")
            
        with h5py.File(f, "r") as H:
            sgaids = H["sgaid"][:]
            
            for i, sgaid in enumerate(sgaids):
                key = (region, int(sgaid))
                if key in index:
                    print(f"WARNING: duplicate key found: {key}")
                index[key] = (f, i)

    print(f"Total unique (region, SGAID) pairs indexed: {len(index):,}")

    return index

def build_jpg_index(base):
    jpg_index = {}

    for kind in ["Model", "Residual", "Image"]:
        directory = os.path.join(base, kind)

        for path in glob.glob(os.path.join(directory, "*.jpg")):
            filename = os.path.basename(path)
            sgaid = filename.split("_", 1)[0]

            jpg_index.setdefault(int(sgaid), {})[kind] = path

    return jpg_index

In [7]:
# Load legacy survey jpgs
def load_jpg_band(row, band):
    """Load exactly one band ('Model', 'Residual', or 'Image') — avoids
    reading and decoding the other two when they're not being displayed."""
    sgaid = int(row["SGAID"])
    path = JPG_INDEX.get(sgaid, {}).get(band)
    if path is None:
        return None
    return np.flipud(mpimg.imread(path))


def load_jpg(row):
    """Kept for backward compatibility with anything still expecting the 3-tuple."""
    return (
        load_jpg_band(row, "Model"),
        load_jpg_band(row, "Residual"),
        load_jpg_band(row, "Image"),
    )

# Taken from SGA.qa
def sdss_rgb(imgs, bands, scales=None, m=0.03, Q=20, mnmx=None, clip=True):
    """Convert a list of band images to an RGB array using an arcsinh stretch.

    Default scaling matches the Legacy Survey viewer::

        g -> Blue  (plane 2, scale 6.0)
        r -> Green (plane 1, scale 3.4)
        i -> Red   (plane 0, scale 3.0)
        z -> Red   (plane 0, scale 2.2)

    For ``['g', 'r', 'i', 'z']`` a band-mixing scheme is used so that
    all four bands contribute to all three color channels. For any
    other combination (e.g. ``['g', 'r', 'z']``) each band maps to a
    single plane.

    Parameters
    ----------
    imgs : :class:`list` of :class:`numpy.ndarray`
        Per-band 2-D image arrays, in the same order as `bands`.
    bands : :class:`list` of :class:`str`
        Band names, from ``{'g', 'r', 'i', 'z'}``.
    scales : :class:`dict`, optional
        Override per-band ``(plane, scale)`` tuples.
    m : :class:`float`
        Additive offset applied before the arcsinh stretch.
    Q : :class:`float` or None
        Arcsinh softening parameter. Set to None to skip the stretch
        entirely (linear scaling only).
    mnmx : :class:`tuple`, optional
        Manual ``(min, max)`` linear clip range, used instead of the
        arcsinh/percentile-style normalization.
    clip : :class:`bool`
        If True, clip the output to ``[0, 1]``.

    Returns
    -------
    :class:`numpy.ndarray`
        ``(H, W, 3)`` float32 RGB array.

    """
    _scales = dict(g=(2, 6.0), r=(1, 3.4), i=(0, 3.0), z=(0, 2.2))
    if scales is not None:
        _scales.update(scales)

    I = 0
    for img, band in zip(imgs, bands):
        _, scale = _scales[band]
        I = I + np.maximum(0, img * scale + m)
    I /= len(bands)
    if Q is not None:
        fI = np.arcsinh(Q * I) / np.sqrt(Q)
        I += (I == 0.) * 1e-6
        I = fI / I

    H, W = I.shape
    rgb = np.zeros((H, W, 3), np.float32)

    if list(bands) == ['g', 'r', 'i', 'z']:
        rgbvec = dict(
            g=(0.,   0.,  0.75),
            r=(0.,   0.5, 0.25),
            i=(0.25, 0.5, 0.),
            z=(0.75, 0.,  0.))
        for img, band in zip(imgs, bands):
            _, scale = _scales[band]
            rf, gf, bf = rgbvec[band]
            if mnmx is None:
                v = (img * scale + m) * I
            else:
                v = ((img * scale + m) - mnmx[0]) / (mnmx[1] - mnmx[0])
            if clip:
                v = np.clip(v, 0, 1)
            rgb[:, :, 0] += rf * v
            rgb[:, :, 1] += gf * v
            rgb[:, :, 2] += bf * v
    else:
        for img, band in zip(imgs, bands):
            plane, scale = _scales[band]
            if mnmx is None:
                imgplane = (img * scale + m) * I
            else:
                imgplane = ((img * scale + m) - mnmx[0]) / (mnmx[1] - mnmx[0])
            if clip:
                imgplane = np.clip(imgplane, 0, 1)
            rgb[:, :, plane] = imgplane

    return rgb

In [8]:
cutout_index = build_cutout_index(SSL_DIR)
JPG_INDEX = build_jpg_index(JPEG_ROOT)
jpg_service = JpgService(JPEG_ROOT, JPG_INDEX)
jpg_service.backfill(store, size_bins, dry_run=False)

Total unique (region, SGAID) pairs indexed: 445,693
0 of 12,427 vetted galaxies are missing JPGs (by size: {}); 0 have no RA/DEC in the catalog.
Queued 0 galaxies. JPGs: 0 pending, 0 done (0 images), 0 failed  (check progress with jpg_service.status())


0

In [9]:
# Check new Image Model Residual downloads while adding new galaxies
jpg_service.status()

'JPGs: 0 pending, 0 done (0 images), 0 failed'

In [10]:
%matplotlib widget

reviewer = ReviewerLogin(
    store=store,
    sdss_rgb_fn=sdss_rgb,
    size_bins=size_bins,
    cutout_index=cutout_index,
    build_cutout_index_fn=build_cutout_index,
    load_jpg_fn=load_jpg,
    load_jpg_band_fn=load_jpg_band,
    ncols=N_COLS,
    n_per_page=N_PER_PAGE,
    n_workers=N_WORKERS,
    jpg_service=jpg_service
)

In [11]:
matrix = confusion_matrix_report(store, 'qshimp')
display(matrix)

,Elliptical,Lenticular,Spiral,Irregular,Bad anchor
Elliptical,2593,101,3,30,19
Lenticular,41,3673,188,23,6
Spiral,0,9,2994,5,0
Irregular,3,13,86,2638,2


In [12]:
v = store.all_votes()
v

,vetter,SGAID,label,proposed,source,size_bin,notes,timestamp
0,nravi,27189,Elliptical,Elliptical,legacy,39-79arcsec,,2026-09-18T15:54:04.222395+00:00
1,nravi,110075,Elliptical,Elliptical,legacy,39-79arcsec,,2026-09-18T15:54:04.222395+00:00
2,nravi,412013,Elliptical,Elliptical,legacy,39-79arcsec,,2026-09-18T15:54:04.222395+00:00
3,nravi,591479,Bad anchor,Elliptical,legacy,39-79arcsec,missing photometry band,2026-09-18T15:54:04.222395+00:00
4,nravi,773955,Elliptical,Elliptical,legacy,39-79arcsec,,2026-09-18T15:54:04.222395+00:00
...,...,...,...,...,...,...,...,...
23874,s,5101630,Irregular,Irregular,legacy,39-79arcsec,,2026-09-21T03:32:34.457824+00:00
23875,s,5101711,Lenticular,Lenticular,consensus:0-39arcsec,0-39arcsec,,2026-10-27T18:32:14.129757+00:00
23876,s,5101929,Elliptical,Elliptical,legacy,39-79arcsec,,2026-09-21T04:45:54.889654+00:00
23877,s,5101994,Irregular,Irregular,consensus:0-39arcsec,0-39arcsec,,2026-10-27T17:38:10.690432+00:00


In [13]:
master

,SGAID,RA,DEC,D26,nravi,qshimp,s
0,570,43.685625,-11.861502,0.645587,NaN,Irregular,Irregular
1,663,52.790040,2.160897,0.908740,NaN,Irregular,Irregular
2,902,87.898192,-60.961305,0.528081,NaN,Irregular,Bad anchor
3,934,94.202777,-14.868604,0.488316,NaN,Irregular,Irregular
4,1462,154.459240,-33.919423,0.864963,NaN,Irregular,Irregular
...,...,...,...,...,...,...,...
12422,5101711,351.026582,14.418256,0.576761,NaN,Lenticular,Lenticular
12423,5101929,133.814796,49.151175,0.879059,NaN,Elliptical,Elliptical
12424,5101994,309.353488,-56.611127,0.468407,NaN,Irregular,Irregular
12425,5102118,359.562165,29.572786,0.790130,NaN,Elliptical,NaN


# To do:
1. Pull high-confidence morphologies to front of large galaxy sample
2. VI large galaxies
3. Repair large galaxy cutouts

# Optional:
1. Use ssl finder to get ringed ellipticals
2. Load bad SGA objects to SGA 2025 issue repository

In [14]:
counts, needed = anchor_progress(store, target=1000)
counts

label,Elliptical,Lenticular,Spiral,Irregular,Bad anchor
size_bin,,,,,
0-39arcsec,1101,1082,1193,1188,11
119-159arcsec,0,1,0,0,0
39-79arcsec,1024,968,971,1103,15
40-79arcsec,0,134,122,0,0
79-119arcsec,2,0,0,0,0


In [15]:
tables = vetting_depth(store)                      # one table each for 1, 2, 3 ... vetters
tables[2]                                          # the "vetted by exactly 2 people" table


=== galaxies vetted by exactly 1 vetter(s) (1,166) ===
cls          Elliptical  Lenticular  Spiral  Irregular  Bad anchor  Disputed  Total
size_bin                                                                           
0-39arcsec            0           0       0          0          10         0     10
39-79arcsec          94         376     224        199           7         0    900
40-79arcsec           0         134     122          0           0         0    256
All bins             94         510     346        199          17         0   1166

=== galaxies vetted by exactly 2 vetter(s) (11,070) ===
cls            Elliptical  Lenticular  Spiral  Irregular  Bad anchor  Disputed  Total
size_bin                                                                             
0-39arcsec           1101        1082    1193       1188           1      2246   6811
119-159arcsec           0           1       0          0           0         0      1
39-79arcsec           781         592  

cls,Elliptical,Lenticular,Spiral,Irregular,Bad anchor,Disputed,Total
size_bin,,,,,,,
0-39arcsec,1101,1082,1193,1188,1,2246,6811
119-159arcsec,0,1,0,0,0,0,1
39-79arcsec,781,592,747,904,8,1224,4256
79-119arcsec,2,0,0,0,0,0,2
All bins,1884,1675,1940,2092,9,3470,11070


In [16]:
### Generate Anchor Catalog for ML Run

In [17]:
store.catalog

,SGAID,RA,DEC,D26
0,1,0.066207,-72.186536,0.592556
1,3,0.114649,-7.881941,1.144977
2,8,0.372656,13.098436,0.848419
3,10,0.408105,-83.236443,0.951040
4,14,0.705979,-66.186566,1.219663
...,...,...,...,...
470620,5102161,169.618007,45.312700,0.626429
470621,5102162,209.594914,47.684057,0.675923
470622,5102163,229.329232,57.268293,0.490805
470623,5102164,172.415764,60.961845,0.620595


In [18]:
g = v.groupby("SGAID").agg(n=("vetter", "nunique"), u=("label", "nunique"), label=("label", "first"),
                               size_bin=("size_bin", lambda x: next((b for b in x if b), "")))
g["cls"] = np.where(g["u"] == 1, g["label"], "Disputed")
g["size_bin"] = g["size_bin"].replace("", "(unbinned)")
g

,n,u,label,size_bin,cls
SGAID,,,,,
570,2,1,Irregular,0-39arcsec,Irregular
663,2,1,Irregular,39-79arcsec,Irregular
902,2,2,Irregular,0-39arcsec,Disputed
934,2,1,Irregular,0-39arcsec,Irregular
1462,2,1,Irregular,39-79arcsec,Irregular
...,...,...,...,...,...
5101711,2,1,Lenticular,0-39arcsec,Lenticular
5101929,2,1,Elliptical,39-79arcsec,Elliptical
5101994,2,1,Irregular,0-39arcsec,Irregular


In [19]:
masterc = master.copy()
masterc['Morphology'] = masterc['SGAID'].map(g['cls'])
masterc['vet_count'] = masterc['SGAID'].map(g['n'])
masterc['size_bin'] = masterc['SGAID'].map(g['size_bin'])
masterc = masterc.drop(columns={'qshimp', 'nravi', 's'})
masterc = masterc[(masterc['vet_count'] != 1) & (masterc['size_bin'] == '0-39arcsec')]
masterc = masterc[(masterc['Morphology'] != 'Disputed')]
masterc = masterc[(masterc['Morphology'] != 'Bad anchor')]
masterc

,SGAID,RA,DEC,D26,Morphology,vet_count,size_bin
0,570,43.685625,-11.861502,0.645587,Irregular,2,0-39arcsec
3,934,94.202777,-14.868604,0.488316,Irregular,2,0-39arcsec
25,3661,10.027860,6.965266,0.600928,Lenticular,2,0-39arcsec
27,3800,55.762602,-17.659035,0.552353,Irregular,2,0-39arcsec
29,4149,102.271752,-25.018473,0.632778,Elliptical,2,0-39arcsec
...,...,...,...,...,...,...,...
12417,5099638,158.284977,-27.454265,0.544060,Irregular,2,0-39arcsec
12418,5101544,0.110465,-72.458574,0.527362,Irregular,2,0-39arcsec
12422,5101711,351.026582,14.418256,0.576761,Lenticular,2,0-39arcsec
12424,5101994,309.353488,-56.611127,0.468407,Irregular,2,0-39arcsec


In [20]:
def sampleMorphologies(df, n=1000, morph_col="Galaxy_Type"):
    sample = df.groupby(morph_col).sample(n)
    return sample

In [21]:
sample = sampleMorphologies(master, morph_col="Morphology")
#sample.to_csv('/global/cfs/cdirs/desicollab/users/qshimp/anchors/anchors_bin0_v2.csv', index=False)

KeyError: 'Morphology'

In [ ]:
import fcntl, errno
p = store.master_dir / ".lock"
a = open(p, "w"); b = open(p, "w")
fcntl.flock(a, fcntl.LOCK_EX | fcntl.LOCK_NB); print("first lock: OK")
try:
    fcntl.flock(fh, fcntl.LOCK_EX | fcntl.LOCK_NB)
    break
except OSError as e:
     if e.errno not in (errno.EAGAIN, errno.EWOULDBLOCK, errno.EACCES):
         print(f"NOTE: filesystem doesn't support flock ({errno.errorcode.get(e.errno, e.errno)}); "
               f"saving without a lock.")
         fh.close(); fh = None
         break
fcntl.flock(a, fcntl.LOCK_UN)